In [8]:
import os
import json
import re
import glob
import pandas as pd
from datetime import datetime

In [10]:
def extract_diagnosis(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data.get('diagnosis', '')

def count_tags(text, tag):
    return len(re.findall(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL))

def has_epds_references(text):
    patterns = [
        r"item \d+", r"question \d+", r"option '[^']+'", r"\d+ points?",
        r"Yes, most of the time", r"Yes, very often", r"Not at all"
    ]
    matches = [bool(re.search(p, text, re.I)) for p in patterns]
    return any(matches)

def has_step_by_step(text):
    patterns = [r"\d+\.\s", r"\*\*Reasoning:\*\*", r"\*\*Next Steps:\*\*", r"therefore", r"because"]
    matches = [bool(re.search(p, text)) for p in patterns]
    return sum(matches) >= 2

def analyze_file(file_path):
    diag = extract_diagnosis(file_path)
    # lấy tên model từ tên file.json
    filename = os.path.basename(file_path)
    model = os.path.splitext(filename)[0]
    model = model.replace('.json', '').replace('result', '')

    return {
        'model': model,
        'sym_count': count_tags(diag, 'sym'),
        'quote_count': count_tags(diag, 'quote'),
        'med_count': count_tags(diag, 'med'),
        'has_epds_ref': 'Yes' if has_epds_references(diag) else 'No',
        'has_steps': 'Yes' if has_step_by_step(diag) else 'No',
        'diagnosis_snippet': diag[:200] + "..." if len(diag) > 200 else diag
    }

# === CHẠY TỰ ĐỘNG CHO 4 FILE ===
files = glob.glob(r"D:\Projects\paper-CT\Maternal-Mental-Health\mental_health_multiagent\datasets\chat_logs\for_evaluate_explainability\*.json")
if len(files) == 0:
    print("Không tìm thấy file JSON nào! Đặt 4 file vào cùng thư mục.")
    exit()

results = [analyze_file(f) for f in files]
df = pd.DataFrame(results)

# === TẠO BẢNG KẾT QUẢ ===
table_md = df[['model', 'sym_count', 'quote_count', 'has_epds_ref', 'has_steps']].to_markdown(index=False)
table_md = table_md.replace('model', '| Model         ').replace('sym_count', '| # `<sym>` ').replace('quote_count', '| # `<quote>` ').replace('has_epds_ref', '| EPDS Ref? ').replace('has_steps', '| Step-by-step? |')
# table_md = table_md.split('\n', 2)[2]  # bỏ 2 dòng đầu

# === XUẤT BÁO CÁO ===
report = f"""# BÁO CÁO ĐÁNH GIÁ EXPLAINABILITY 

**Số file phân tích**: {len(files)}\n
**Tiêu chuẩn**: `<sym>`, `<quote>`, EPDS references, Step-by-step logic\n

## Bảng Kết Quả 

{table_md}

"""

print(report)

# BÁO CÁO ĐÁNH GIÁ EXPLAINABILITY 

**Số file phân tích**: 4

**Tiêu chuẩn**: `<sym>`, `<quote>`, EPDS references, Step-by-step logic


## Bảng Kết Quả 

| | Model                                               |   | # `<sym>`  |   | # `<quote>`  | | EPDS Ref?    | | Step-by-step? |   |
|:-------------------------------------------|------------:|--------------:|:---------------|:------------|
| chat_20251006_234015_batch_1_openai        |           3 |             1 | Yes            | Yes         |
| chat_20251008_062310_batch_1_moonshotai    |          15 |             1 | Yes            | Yes         |
| chat_20251012_145929_batch_1_llama-4       |           3 |             0 | Yes            | Yes         |
| chat_20251016_132646_batch_1_llama-3.3-70b |          14 |             0 | Yes            | Yes         |


